# Introduction

This notebook presents an implementaiton of the **inference pipeline for a GraphRAG via Lexical Search**, using:
- Tech Stack: Neo4j and LangChain
- Query Method: Lexical Search via Cypher query language (not vector search/semantic similarity search)

**Setting up**

You will need to have a running Neo4j instance, which there are some ways to do that. To replicate the results here, you will need:
1. create a [free Neo4j database instance in their Aura cloud service](https://neo4j.com/cloud/platform/aura-graph-database/) and
2. import the [FDA Adverse Event Reporting System (FAERS)](https://open.fda.gov/data/faers/) database from the healthcare-analytics guide that contains information on adverse event and medication error reports submitted to [FDA](https://www.fda.gov/).
*This approach makes the inference pipeline independent of the indexing pipeline. So, you want skip the graph creation and jump directly to the RAG part, you can start with this notebook*.

**Sources**
1. [DeepLearning.AI | Knowledge Graphs for RAG](https://learn.deeplearning.ai/courses/knowledge-graphs-rag/lesson/8/chatting-with-the-knowledge-graph)
2. [LangChain | Neo4j](https://python.langchain.com/v0.2/docs/integrations/graphs/neo4j_cypher/)
3. [LangChain | Chains - GraphCypherQAChain](https://python.langchain.com/v0.1/docs/modules/chains/)

In [1]:
! pip install --quiet \
              langchain==0.2.14 \
              langchain-openai==0.1.20 \
              langchain-community==0.2.10 \
              neo4j-graphrag==1.2.0 \
              python-dotenv==1.0.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.8/997.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import os

from dotenv import load_dotenv, find_dotenv
from langchain.chains import GraphCypherQAChain
from langchain.prompts.prompt import PromptTemplate
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI

In [3]:
use_google_drive_api = True
google_drive_path = 'gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain'

if(use_google_drive_api):
  from google.colab import drive
  drive.mount('/content/gdrive/', force_remount=True)
  # Changing the current working directory to the notebook folder
  os.chdir(google_drive_path)
  print(os.getcwd())

Mounted at /content/gdrive/
/content/gdrive/My Drive/Colab Notebooks/Toolkit-NLP/GraphRAG_with_Neo4j_and_LangChain


In [4]:
# Enverionment variables
_ = load_dotenv(find_dotenv(), override=True) # read local .env file

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USERNAME = os.environ["NEO4J_USERNAME"]
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

# 1. Import Graph

In [5]:
kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

In [7]:
kg.refresh_schema()
print(kg.schema)

Node properties:
AgeGroup {id: INTEGER, ageGroup: STRING}
Case {id: INTEGER, age: FLOAT, ageUnit: STRING, eventDate: DATE_TIME, gender: STRING, primaryid: INTEGER, reportDate: DATE_TIME, reporterOccupation: STRING}
Drug {id: INTEGER, name: STRING, primarySubstabce: STRING}
Manufacturer {id: INTEGER, manufacturerName: STRING}
Outcome {id: INTEGER, code: STRING, outcome: STRING}
Reaction {id: INTEGER, description: STRING, embedding: LIST}
ReportSource {id: INTEGER, name: STRING, code: STRING}
Therapy {id: INTEGER, primaryid: INTEGER}
Relationship properties:
IS_CONCOMITANT {indication: STRING, doseAmount: STRING, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
IS_PRIMARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: BOOLEAN, route: STRING}
IS_SECONDARY_SUSPECT {indication: STRING, doseAmount: FLOAT, doseUnit: STRING, drugSequence: INTEGER, route: STRING}
PRESCRIBED {drugSequence: INTEGER, endYear: INTEGER, startYear: INTEGER}
The relationships:
(

## 1.1. Querying the graph

In [8]:
# Returns the side effects using the HAS_REACTION relationship between the Case and Reaction nodes.

query = """
MATCH (c:Case)-[h:HAS_REACTION]->(r:Reaction)
RETURN c,h,r
LIMIT 1
"""

kg.query(query)

[{'c': {'ageUnit': 'YR',
   'gender': 'F',
   'reportDate': neo4j.time.DateTime(2021, 8, 27, 0, 0, 0, 0, tzinfo=<UTC>),
   'reporterOccupation': 'Physician',
   'id': 0,
   'primaryid': 100654764,
   'age': 56.0,
   'eventDate': neo4j.time.DateTime(2014, 3, 25, 0, 0, 0, 0, tzinfo=<UTC>)},
  'h': ({'ageUnit': 'YR',
    'gender': 'F',
    'reportDate': neo4j.time.DateTime(2021, 8, 27, 0, 0, 0, 0, tzinfo=<UTC>),
    'reporterOccupation': 'Physician',
    'id': 0,
    'primaryid': 100654764,
    'age': 56.0,
    'eventDate': neo4j.time.DateTime(2014, 3, 25, 0, 0, 0, 0, tzinfo=<UTC>)},
   'HAS_REACTION',
   {'description': 'Pulmonary embolism',
    'id': 4454,
    'embedding': [-0.030448319390416145,
     -0.014532151632010937,
     0.01440633274614811,
     -0.01784120500087738,
     -0.029617909342050552,
     0.017778294160962105,
     -0.027453813701868057,
     -0.015048012137413025,
     -0.02219455875456333,
     -0.007719026412814856,
     -0.007599497679620981,
     0.0222197230905

In [9]:
# find the number of cases for each age group.

query = """
MATCH (a:AgeGroup)<-[:FALLS_UNDER]-(c:Case)
RETURN a.ageGroup as ageGroupName, count(c) as total_cases
ORDER BY total_cases DESC
"""

kg.query(query)

[{'ageGroupName': 'Adult', 'total_cases': 2103},
 {'ageGroupName': 'Elderly', 'total_cases': 1865},
 {'ageGroupName': 'Adolescent', 'total_cases': 218},
 {'ageGroupName': 'Child', 'total_cases': 101},
 {'ageGroupName': 'Infant', 'total_cases': 20}]

In [10]:
 # find the number of cases which resulted in Death for each age group.

query = """
MATCH (a:AgeGroup)<-[:FALLS_UNDER]-(c:Case)
MATCH (c)-[:RESULTED_IN]->(o:Outcome{outcome:"Death"})
RETURN a.ageGroup as ageGroupName, count(c) as total_death_cases
ORDER BY total_death_cases DESC
"""

kg.query(query)

[{'ageGroupName': 'Elderly', 'total_death_cases': 212},
 {'ageGroupName': 'Adult', 'total_death_cases': 107},
 {'ageGroupName': 'Adolescent', 'total_death_cases': 6},
 {'ageGroupName': 'Child', 'total_death_cases': 3},
 {'ageGroupName': 'Infant', 'total_death_cases': 1}]

In [11]:
# returns the number of cases whose outcome was Death for Opdivo and LONSURF drugs

query = """
MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome),
      (c)-[:IS_PRIMARY_SUSPECT]->(d)
WHERE o.outcome = "Death" and d.name in ["OPDIVO", "LONSURF"]
RETURN d.name as drug, o.outcome as outcome, count(c) as cases
ORDER BY cases DESC
"""

kg.query(query)


[{'drug': 'LONSURF', 'outcome': 'Death', 'cases': 6},
 {'drug': 'OPDIVO', 'outcome': 'Death', 'cases': 4}]

# 2. Chating with the graph

In [12]:
# Instantiate an LLM

llm_large = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    temperature=0.4,
    model_name="gpt-4o",
)

llm_mini = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    temperature=0.4,
    model_name="gpt-4o-mini",
)

In [13]:
chain = GraphCypherQAChain.from_llm(
    llm=llm_mini, graph=kg, verbose=True
)

In [14]:
query = "What are the number of cases for each age group?"
chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (a:AgeGroup)<-[:FALLS_UNDER]-(c:Case)
RETURN a.ageGroup, COUNT(c) AS numberOfCases

Full Context:
[{'a.ageGroup': 'Adult', 'numberOfCases': 2103}, {'a.ageGroup': 'Elderly', 'numberOfCases': 1865}, {'a.ageGroup': 'Adolescent', 'numberOfCases': 218}, {'a.ageGroup': 'Child', 'numberOfCases': 101}, {'a.ageGroup': 'Infant', 'numberOfCases': 20}]

> Finished chain.


{'query': 'What are the number of cases for each age group?',
 'result': 'The number of cases for each age group is as follows: Adults have 2,103 cases, Elderly have 1,865 cases, Adolescents have 218 cases, Children have 101 cases, and Infants have 20 cases.'}

In [ ]:
query = "What are the number of cases which resulted in Death for each age group in 2013?"
chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (a:AgeGroup)<-[:FALLS_UNDER]-(c:Case)-[:RESULTED_IN]->(o:Outcome)
WHERE o.outcome = 'Death' AND c.eventDate >= date('2013-01-01') AND c.eventDate < date('2014-01-01')
RETURN a.ageGroup, COUNT(c) AS numberOfCases

Full Context:
[]

> Finished chain.


{'query': 'What are the number of cases which resulted in Death for each age group in 2013?',
 'result': "I don't know the answer."}

In [ ]:
kg.query(
    """
    MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome {outcome: 'Death'}),
          (c)-[:FALLS_UNDER]->(a:AgeGroup)
    WHERE c.eventDate.year = 2013
    RETURN a.ageGroup AS AgeGroup, COUNT(c) AS NumberOfCases
    """
)

[{'AgeGroup': 'Elderly', 'NumberOfCases': 1},
 {'AgeGroup': 'Adult', 'NumberOfCases': 3}]

### Defining different LLMs for QA and Cypher generation

In [ ]:
chain = GraphCypherQAChain.from_llm(
    qa_llm=llm_mini, cypher_llm=llm_large, graph=kg, verbose=True
)

In [ ]:
query = "What are the number of cases which resulted in Death for each age group in 2013?"
chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome {outcome: 'Death'}), 
      (c)-[:FALLS_UNDER]->(a:AgeGroup)
WHERE c.eventDate.year = 2013
RETURN a.ageGroup AS AgeGroup, COUNT(c) AS NumberOfCases

Full Context:
[{'AgeGroup': 'Elderly', 'NumberOfCases': 1}, {'AgeGroup': 'Adult', 'NumberOfCases': 3}]

> Finished chain.


{'query': 'What are the number of cases which resulted in Death for each age group in 2013?',
 'result': 'In 2013, there was 1 case resulting in death for the elderly age group and 3 cases for the adult age group.'}

In [ ]:
# returns the number of cases whose outcome was Death for each drug.

query = "What are the number of cases whose outcome was Death for Opdivo?"
chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome {outcome: 'Death'}),
      (c)-[:IS_PRIMARY_SUSPECT|:IS_SECONDARY_SUSPECT|:IS_CONCOMITANT]->(d:Drug {name: 'Opdivo'})
RETURN count(c) AS numberOfCases



Full Context:
[{'numberOfCases': 0}]

> Finished chain.


{'query': 'What are the number of cases whose outcome was Death for Opdivo?',
 'result': 'There are 0 cases whose outcome was Death for Opdivo.'}

In [ ]:
kg.query(
    """
    MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome)
    MATCH (c)-[:IS_PRIMARY_SUSPECT]->(d)
    WHERE o.outcome = "Death" and toLower(d.name) = "opdivo"
    RETURN d.name as drug, o.outcome as outcome, count(c) as cases
    ORDER BY cases DESC
    """
)

[{'drug': 'OPDIVO', 'outcome': 'Death', 'cases': 4}]

### Use in-context learning (few shot learning) to improve the system

In [ ]:
# https://github.com/langchain-ai/langchain/blob/master/libs/community/langchain_community/chains/graph_qa/prompts.py

CYPHER_GENERATION_TEMPLATE = """Task:Generate Cypher statement to query a graph database.
Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.
Schema:
{schema}
Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

Examples: Here are a few examples of generated Cypher
statements for particular questions:

# What are the number of cases whose outcome was Death for Opdivo?
MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome)
MATCH (c)-[:IS_PRIMARY_SUSPECT]->(d)
WHERE o.outcome = "Death" and toLower(d.name) = "opdivo"
RETURN d.name as drug, o.outcome as outcome, count(c) as cases
ORDER BY cases DESC

The question is:
{question}"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)

In [ ]:
chain = GraphCypherQAChain.from_llm(
    qa_llm=llm_mini, cypher_llm=llm_large, graph=kg, cypher_prompt=CYPHER_GENERATION_PROMPT, verbose=True
)

In [ ]:
# returns the number of cases whose outcome was Death for Opdivo and LONSURF drugs

query = "What are the number of cases whose outcome was Death for Opdivo and LONSURF?"
chain.invoke(query)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Case)-[:RESULTED_IN]->(o:Outcome)
MATCH (c)-[:IS_PRIMARY_SUSPECT]->(d)
WHERE o.outcome = "Death" AND toLower(d.name) IN ["opdivo", "lonsurf"]
RETURN d.name AS drug, o.outcome AS outcome, count(c) AS cases
ORDER BY cases DESC
Full Context:
[{'drug': 'LONSURF', 'outcome': 'Death', 'cases': 6}, {'drug': 'OPDIVO', 'outcome': 'Death', 'cases': 4}]

> Finished chain.


{'query': 'What are the number of cases whose outcome was Death for Opdivo and LONSURF?',
 'result': 'There were 6 cases with the outcome of Death for LONSURF and 4 cases for OPDIVO.'}